# Parallel File Processor – Data Science Project
**Platform:** Google Colab  
**Language:** Python  
**Libraries:** Pandas, NumPy, Matplotlib, Seaborn, concurrent.futures

## Project overview
This project reads multiple CSV files, cleans the data, calculates useful statistics, combines the results, and compares sequential processing with parallel processing.

### Objectives
1. Create or upload multiple CSV files.
2. Read and clean files automatically.
3. Process files concurrently using `ThreadPoolExecutor`.
4. Generate data-science summaries and visualizations.
5. Compare sequential and parallel execution time.

**Note:** This version uses threads to process multiple file-reading and data-preparation tasks concurrently. It is especially useful for I/O-heavy workflows. The timing result depends on the machine and dataset size.

## 1. Import required libraries

In [ ]:
import os
import time
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import files

print("All libraries imported successfully!")

## 2. Create sample CSV files
Run this cell to create sample sales datasets automatically. You can skip this and upload your own CSV files in the next section.

In [ ]:
DATA_DIR = "/content/parallel_data"
os.makedirs(DATA_DIR, exist_ok=True)

def create_sample_files(number_of_files=5, rows_per_file=2000):
    rng = np.random.default_rng(42)
    for file_no in range(1, number_of_files + 1):
        df = pd.DataFrame({
            "Transaction_ID": [f"T{file_no}_{i}" for i in range(rows_per_file)],
            "Date": pd.date_range("2025-01-01", periods=rows_per_file, freq="h"),
            "Product": rng.choice(["Laptop", "Mobile", "Tablet", "Headphones", "Keyboard"], rows_per_file),
            "Category": rng.choice(["Electronics", "Accessories"], rows_per_file),
            "Quantity": rng.integers(1, 6, rows_per_file),
            "Unit_Price": rng.uniform(300, 50000, rows_per_file).round(2),
            "Region": rng.choice(["North", "South", "East", "West"], rows_per_file),
            "Customer_Rating": rng.uniform(1, 5, rows_per_file).round(1)
        })
        # Add a few missing values to demonstrate data cleaning
        if rows_per_file >= 20:
            df.loc[rng.choice(rows_per_file, size=5, replace=False), "Customer_Rating"] = np.nan
        df["Sales"] = df["Quantity"] * df["Unit_Price"]
        df.to_csv(os.path.join(DATA_DIR, f"sales_{file_no}.csv"), index=False)

create_sample_files()
print("Sample CSV files created:")
print("\n".join(os.listdir(DATA_DIR)))

## 3. Upload your own CSV files (optional)
If you want to use your own data, run this cell and select multiple CSV files. They will be copied into the project folder. If you use the sample data, you can skip this cell.

In [ ]:
# Optional: upload CSV files from your computer
uploaded = files.upload()

for filename, content in uploaded.items():
    if filename.lower().endswith(".csv"):
        destination = os.path.join(DATA_DIR, os.path.basename(filename))
        with open(destination, "wb") as f:
            f.write(content)

print("CSV files available:", len(glob.glob(os.path.join(DATA_DIR, "*.csv"))))

## 4. Define the file-processing function
This function reads one CSV file, removes duplicate rows, fills missing numeric values with the median, fills missing text values with `Unknown`, calculates sales if needed, and returns a cleaned dataset plus a summary.

In [ ]:
def process_one_file(file_path):
    start = time.perf_counter()

    df = pd.read_csv(file_path)
    original_rows = len(df)

    # Remove duplicate rows
    df = df.drop_duplicates()

    # Clean missing values
    numeric_columns = df.select_dtypes(include=np.number).columns
    text_columns = df.select_dtypes(include=["object"]).columns

    for column in numeric_columns:
        df[column] = df[column].fillna(df[column].median())

    for column in text_columns:
        df[column] = df[column].fillna("Unknown")

    # Create Sales column if it does not already exist
    if "Sales" not in df.columns and {"Quantity", "Unit_Price"}.issubset(df.columns):
        df["Sales"] = df["Quantity"] * df["Unit_Price"]

    # Add source filename
    df["Source_File"] = os.path.basename(file_path)

    summary = {
        "File": os.path.basename(file_path),
        "Original_Rows": original_rows,
        "Cleaned_Rows": len(df),
        "Duplicates_Removed": original_rows - len(df),
        "Missing_Values_After_Cleaning": int(df.isna().sum().sum()),
        "Processing_Time_Seconds": round(time.perf_counter() - start, 4)
    }

    return df, summary

print("Processing function is ready.")

## 5. Find CSV files and process sequentially
Sequential processing handles one file after another. We use it as a baseline for comparison.

In [ ]:
file_paths = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))

if not file_paths:
    raise FileNotFoundError("No CSV files found. Run the sample-data cell or upload CSV files.")

print("Files found:", len(file_paths))

sequential_start = time.perf_counter()
sequential_results = [process_one_file(path) for path in file_paths]
sequential_time = time.perf_counter() - sequential_start

sequential_data = [item[0] for item in sequential_results]
sequential_summaries = [item[1] for item in sequential_results]

print(f"Sequential processing time: {sequential_time:.4f} seconds")

## 6. Process files in parallel
`ThreadPoolExecutor` runs multiple file-processing tasks concurrently. The number of workers is limited to 4 for a reasonable Colab workload.

In [ ]:
parallel_start = time.perf_counter()

parallel_results = []
max_workers = min(4, len(file_paths))

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(process_one_file, path): path for path in file_paths}

    for future in as_completed(futures):
        path = futures[future]
        try:
            parallel_results.append(future.result())
            print("Completed:", os.path.basename(path))
        except Exception as error:
            print("Error processing", os.path.basename(path), ":", error)

parallel_time = time.perf_counter() - parallel_start

parallel_data = [item[0] for item in parallel_results]
parallel_summaries = [item[1] for item in parallel_results]

print(f"Parallel processing time: {parallel_time:.4f} seconds")
print("Files successfully processed:", len(parallel_data))

## 7. Combine all cleaned data and save output files

In [ ]:
if not parallel_data:
    raise RuntimeError("No files were processed successfully.")

final_df = pd.concat(parallel_data, ignore_index=True)
summary_df = pd.DataFrame(parallel_summaries)

OUTPUT_DIR = "/content/parallel_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

cleaned_path = os.path.join(OUTPUT_DIR, "combined_cleaned_data.csv")
summary_path = os.path.join(OUTPUT_DIR, "processing_summary.csv")

final_df.to_csv(cleaned_path, index=False)
summary_df.to_csv(summary_path, index=False)

print("Combined dataset shape:", final_df.shape)
display(final_df.head())
display(summary_df)
print("Saved:", cleaned_path)
print("Saved:", summary_path)

## 8. Data science analysis

In [ ]:
print("Dataset information:")
final_df.info()

print("\nDescriptive statistics:")
display(final_df.describe(include="all").T)

if "Sales" in final_df.columns:
    print("\nTotal sales:", round(final_df["Sales"].sum(), 2))
    print("Average sales per row:", round(final_df["Sales"].mean(), 2))

if "Product" in final_df.columns and "Sales" in final_df.columns:
    product_sales = (final_df.groupby("Product")["Sales"]
                     .sum().sort_values(ascending=False).reset_index())
    print("\nSales by product:")
    display(product_sales)

if "Region" in final_df.columns and "Sales" in final_df.columns:
    region_sales = (final_df.groupby("Region")["Sales"]
                    .sum().sort_values(ascending=False).reset_index())
    print("\nSales by region:")
    display(region_sales)

## 9. Visualize the results

In [ ]:
sns.set_theme()

# Chart 1: Sales by product
if {"Product", "Sales"}.issubset(final_df.columns):
    plt.figure(figsize=(9, 5))
    product_order = final_df.groupby("Product")["Sales"].sum().sort_values(ascending=False).index
    sns.barplot(data=final_df, x="Product", y="Sales", estimator=sum, order=product_order, errorbar=None)
    plt.title("Total Sales by Product")
    plt.xlabel("Product")
    plt.ylabel("Total Sales")
    plt.xticks(rotation=25)
    plt.tight_layout()
    plt.show()

# Chart 2: Sales by region
if {"Region", "Sales"}.issubset(final_df.columns):
    plt.figure(figsize=(8, 5))
    region_totals = final_df.groupby("Region", as_index=False)["Sales"].sum()
    sns.barplot(data=region_totals, x="Region", y="Sales", errorbar=None)
    plt.title("Total Sales by Region")
    plt.xlabel("Region")
    plt.ylabel("Total Sales")
    plt.tight_layout()
    plt.show()

# Chart 3: Distribution of sales
if "Sales" in final_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(final_df["Sales"], bins=30, kde=True)
    plt.title("Distribution of Sales")
    plt.xlabel("Sales")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

## 10. Compare sequential and parallel performance

In [ ]:
comparison_df = pd.DataFrame({
    "Method": ["Sequential", "Parallel"],
    "Execution_Time_Seconds": [sequential_time, parallel_time]
})

display(comparison_df)

plt.figure(figsize=(7, 5))
sns.barplot(data=comparison_df, x="Method", y="Execution_Time_Seconds", errorbar=None)
plt.title("Sequential vs Parallel Processing Time")
plt.ylabel("Execution Time (seconds)")
plt.xlabel("Processing Method")
plt.tight_layout()
plt.show()

if parallel_time > 0:
    speedup = sequential_time / parallel_time
    print(f"Speedup ratio (sequential / parallel): {speedup:.2f}x")

if parallel_time < sequential_time:
    print("Parallel processing was faster for this run.")
else:
    print("Sequential processing was faster or similar for this run.")
    print("This can happen with small files because parallel tasks have overhead.")

## 11. Download project results

In [ ]:
from google.colab import files

files.download(cleaned_path)
files.download(summary_path)

## 12. Conclusion
The Parallel File Processor project demonstrates how multiple CSV files can be cleaned and analyzed using Python. It uses concurrent processing to handle multiple files, combines the cleaned data, calculates useful statistics, creates visualizations, and compares sequential and parallel execution time.

### Future improvements
- Support Excel and JSON files.
- Add outlier detection and more advanced data cleaning.
- Build a Streamlit dashboard.
- Use process-based parallelism for CPU-heavy tasks.
- Add logging and detailed error reports.

### Short project explanation for classroom
“My project is Parallel File Processor. It processes multiple CSV files at the same time. First, it reads the files and cleans missing and duplicate values. Then it combines the data, performs data analysis, and creates graphs. Finally, it compares sequential and parallel processing time. This project shows how parallel computing can help automate data-processing tasks.”